In [1]:
pip install pygame

In [2]:
import pygame
import sys
import random
import urllib.request
import io
import os
import tempfile

# Импорт конфигурации
try:
    from config import *
except ImportError:
    print("Файл не найден!")

# Инициализация Pygame и аудио
pygame.init()
pygame.mixer.init(frequency=44100, size=-16, channels=2, buffer=512)

screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Snake Game with Online Sound")
clock = pygame.time.Clock()

# Класс аудио
class AudioSystem:
    def __init__(self):
        self.sound_enabled = True
        self.eat_sound = None
        self.death_sound = None
        self.move_sound = None
        self.menu_sound = None
        
        # URL звуковых файлов из интернета
        self.sound_urls = {
            "eat": "https://assets.mixkit.co/sfx/preview/mixkit-unlock-game-notification-253.mp3",
            "death": "https://assets.mixkit.co/sfx/preview/mixkit-retro-game-over-1947.mp3",
            "move": "https://assets.mixkit.co/sfx/preview/mixkit-game-ball-tap-2073.mp3",
            "menu": "https://assets.mixkit.co/sfx/preview/mixkit-select-click-1109.mp3"
        }
        
        # Альтернативные звуки 
        self.alternative_sounds = {
            "eat": "https://assets.mixkit.co/sfx/preview/mixkit-arcade-game-jump-coin-216.mp3",
            "death": "https://assets.mixkit.co/sfx/preview/mixkit-wrong-answer-fail-notification-946.mp3",
            "move": "https://assets.mixkit.co/sfx/preview/mixkit-plastic-bubble-click-1124.mp3",
            "menu": "https://assets.mixkit.co/sfx/preview/mixkit-click-button-1108.mp3"
        }
        
        # Локальные имена файлов
        self.local_files = {
            "eat": "eat_sound.mp3",
            "death": "death_sound.mp3",
            "move": "move_sound.mp3",
            "menu": "menu_sound.mp3"
        }
        
        print("=" * 50)
        print("ИНИЦИАЛИЗАЦИЯ АУДИО СИСТЕМЫ")
        print("=" * 50)
        
        # Создаем папку для звуков если её нет
        self.sounds_dir = "game_sounds"
        if not os.path.exists(self.sounds_dir):
            os.makedirs(self.sounds_dir)
            print(f"Создана папка для звуков: {self.sounds_dir}")
        
        # Пытаемся загрузить или скачать звуки
        self.load_or_download_sounds()
        
        print("=" * 50)
    
    def download_sound(self, sound_name, url):
        try:
            print(f"Скачиваю звук {sound_name}...")
            
            # Скачиваем файл
            response = urllib.request.urlopen(url)
            sound_data = response.read()
            
            # Сохраняем во временный файл
            temp_file = tempfile.NamedTemporaryFile(delete=False, suffix='.mp3')
            temp_file.write(sound_data)
            temp_file.close()
            
            # Загружаем звук в pygame
            sound = pygame.mixer.Sound(temp_file.name)
            
            # Сохраняем в локальную папку для повторного использования
            local_path = os.path.join(self.sounds_dir, self.local_files[sound_name])
            with open(local_path, 'wb') as f:
                f.write(sound_data)
            
            print(f"✓ Звук {sound_name} успешно скачан")
            return sound
            
        except Exception as e:
            print(f"✗ Ошибка скачивания звука {sound_name}: {e}")
            return None
    
    def load_local_sound(self, sound_name):
        local_path = os.path.join(self.sounds_dir, self.local_files[sound_name])
        
        if os.path.exists(local_path):
            try:
                print(f"Загружаю локальный звук {sound_name}...")
                sound = pygame.mixer.Sound(local_path)
                print(f"✓ Локальный звук {sound_name} загружен")
                return sound
            except Exception as e:
                print(f"✗ Ошибка загрузки локального звука {sound_name}: {e}")
                return None
        return None
    
    def create_fallback_sound(self, sound_name):
        import numpy as np
        
        sample_rate = 44100
        
        if sound_name == "eat":
            # Веселый звук поедания
            duration = 0.5
            t = np.linspace(0, duration, int(sample_rate * duration), False)
            tone1 = 0.3 * np.sin(2 * np.pi * 523.25 * t)  # До
            tone2 = 0.2 * np.sin(2 * np.pi * 659.25 * t)  # Ми
            envelope = np.exp(-4 * t)
            sound_data = (tone1 + tone2) * envelope
            
        elif sound_name == "death":
            # Грустный звук смерти
            duration = 1.0
            t = np.linspace(0, duration, int(sample_rate * duration), False)
            freq = 392 * np.exp(-2 * t)  # Плавное снижение частоты
            sound_data = 0.4 * np.sin(2 * np.pi * freq * t) * np.exp(-5 * t)
            
        elif sound_name == "move":
            # Тихий звук движения
            duration = 0.1
            t = np.linspace(0, duration, int(sample_rate * duration), False)
            sound_data = 0.2 * np.sin(2 * np.pi * 200 * t) * np.exp(-50 * t)
            
        elif sound_name == "menu":
            # Клик для меню
            duration = 0.2
            t = np.linspace(0, duration, int(sample_rate * duration), False)
            sound_data = 0.3 * np.sin(2 * np.pi * 440 * t) * np.exp(-10 * t)
            
        else:
            return None
        
        # Нормализация
        sound_data = np.int16(sound_data / np.max(np.abs(sound_data)) * 32767)
        
        # Создание временного файла
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix='.wav')
        
        # Сохраняем как WAV файл
        import wave
        with wave.open(temp_file.name, 'w') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(sample_rate)
            wav_file.writeframes(sound_data.astype(np.int16).tobytes())
        
        temp_file.close()
        
        try:
            sound = pygame.mixer.Sound(temp_file.name)
            print(f"✓ Создан резервный звук {sound_name}")
            return sound
        except:
            return None
    
    def load_or_download_sounds(self):
        for sound_name in ["eat", "death", "move", "menu"]:
            # Пытаемся загрузить из локального файла
            sound = self.load_local_sound(sound_name)
            
            # Если нет локального файла, пытаемся скачать
            if not sound:
                sound = self.download_sound(sound_name, self.sound_urls[sound_name])
            
            # Если скачивание не удалось, пробуем альтернативный URL
            if not sound and sound_name in self.alternative_sounds:
                print(f"Пробую альтернативный URL для {sound_name}...")
                sound = self.download_sound(sound_name, self.alternative_sounds[sound_name])
            
            # Если все еще нет звука, создаем резервный
            if not sound:
                print(f"Создаю резервный звук для {sound_name}...")
                sound = self.create_fallback_sound(sound_name)
            
            # Сохраняем звук
            if sound:
                setattr(self, f"{sound_name}_sound", sound)
                print(f"Звук '{sound_name}' готов к использованию")
            else:
                print(f"⚠ Не удалось загрузить звук '{sound_name}'")
        
        print("=" * 50)
        print("СТАТУС ЗВУКОВ:")
        print(f"Поедание еды: {'✓' if self.eat_sound else '✗'}")
        print(f"Смерть: {'✓' if self.death_sound else '✗'}")
        print(f"Движение: {'✓' if self.move_sound else '✗'}")
        print(f"Меню: {'✓' if self.menu_sound else '✗'}")
        print("=" * 50)
    
    def play_eat_sound(self):
        if self.sound_enabled and self.eat_sound:
            try:
                self.eat_sound.play()
            except Exception as e:
                print(f"Ошибка воспроизведения звука поедания: {e}")
    
    def play_death_sound(self):
        if self.sound_enabled and self.death_sound:
            try:
                self.death_sound.play()
            except:
                pass
    
    def play_move_sound(self):
        if self.sound_enabled and self.move_sound:
            try:
                # Делаем звук движения тише
                self.move_sound.set_volume(0.3)
                self.move_sound.play()
            except:
                pass
    
    def play_menu_sound(self):
        if self.sound_enabled and self.menu_sound:
            try:
                self.menu_sound.play()
            except:
                pass
    
    def toggle_sound(self):
        self.sound_enabled = not self.sound_enabled
        status = "ВКЛЮЧЕНЫ" if self.sound_enabled else "ВЫКЛЮЧЕНЫ"
        print(f"Звуки: {status}")
        return self.sound_enabled

# Класс змейки
class Snake:
    def __init__(self, audio_system):
        self.audio = audio_system
        self.reset()

    # Сброс к первоначальному состоянию
    def reset(self):
        self.length = INITIAL_SNAKE_LENGTH
        self.positions = [(GRID_WIDTH // 2, GRID_HEIGHT // 2)]
        for i in range(1, self.length):
            self.positions.append((self.positions[0][0] - i, self.positions[0][1]))
        self.direction = (1, 0)
        self.score = 0
        self.grow_pending = 0
        self.alive = True
        self.last_move_time = 0
        self.move_sound_cooldown = 100  # мс между звуками движения
    
    def get_head_position(self):
        return self.positions[0]

    # Изменение направления
    def turn(self, point):
        if self.length > 1 and (point[0] * -1, point[1] * -1) == self.direction:
            return
        else:
            self.direction = point
            # Звук поворота
            if self.alive:
                self.audio.play_menu_sound()

    # Движение Змейки с проверкой столкновений
    def move(self):
        if not self.alive:
            return False
            
        current_time = pygame.time.get_ticks()
        
        # Звук движения (с задержкой)
        if current_time - self.last_move_time > self.move_sound_cooldown:
            self.audio.play_move_sound()
            self.last_move_time = current_time
            
        head = self.get_head_position()
        x, y = self.direction
        new_head = ((head[0] + x), (head[1] + y)) 
        
        # Проверка столкновения со стенами
        if (new_head[0] < 0 or new_head[0] >= GRID_WIDTH or 
            new_head[1] < 0 or new_head[1] >= GRID_HEIGHT):
            self.alive = False
            self.audio.play_death_sound()
            return False
        
        # Проверка столкновения с собой
        if new_head in self.positions[1:]:
            self.alive = False
            self.audio.play_death_sound()
            return False
        
        self.positions.insert(0, new_head)
        
        if self.grow_pending > 0:
            self.grow_pending -= 1
        else:
            self.positions.pop()
        
        return True

    # Увеличение Змейки с звуком
    def grow(self):
        self.grow_pending += 1
        self.score += 10
        self.audio.play_eat_sound()  
    
    def draw(self, surface):
        for i, p in enumerate(self.positions):
            # Если змейка мертва, рисуем красным цветом
            if not self.alive:
                color = (255, 100, 100) if i == 0 else (200, 80, 80)
            else:
                color = GREEN if i == 0 else DARK_GREEN
                
            rect = pygame.Rect((p[0] * GRID_SIZE, p[1] * GRID_SIZE), (GRID_SIZE, GRID_SIZE))
            pygame.draw.rect(surface, color, rect)
            pygame.draw.rect(surface, BLACK, rect, 1)
            
        # Рисуем стенки поля
        self.draw_walls(surface)

    # Отрисовка стенок игрового поля
    def draw_walls(self, surface):
        wall_color = (100, 100, 200)  
        
        # Верхняя стенка
        pygame.draw.rect(surface, wall_color, 
                        pygame.Rect(0, 0, SCREEN_WIDTH, GRID_SIZE))
        
        # Нижняя стенка
        pygame.draw.rect(surface, wall_color, 
                        pygame.Rect(0, SCREEN_HEIGHT - GRID_SIZE, SCREEN_WIDTH, GRID_SIZE))
        
        # Левая стенка
        pygame.draw.rect(surface, wall_color, 
                        pygame.Rect(0, 0, GRID_SIZE, SCREEN_HEIGHT))
        
        # Правая стенка
        pygame.draw.rect(surface, wall_color, 
                        pygame.Rect(SCREEN_WIDTH - GRID_SIZE, 0, GRID_SIZE, SCREEN_HEIGHT))

# Класс еды
class Food:
    def __init__(self):
        self.position = (0, 0)
        self.randomize_position()

    # Случайное размещение еды 
    def randomize_position(self, snake_positions=None):
        while True:
            # Еда появляется только внутри игрового поля 
            self.position = (random.randint(1, GRID_WIDTH - 2), 
                           random.randint(1, GRID_HEIGHT - 2))
            if snake_positions is None or self.position not in snake_positions:
                break
    
    def draw(self, surface):
        rect = pygame.Rect((self.position[0] * GRID_SIZE, self.position[1] * GRID_SIZE), 
                          (GRID_SIZE, GRID_SIZE))
        pygame.draw.rect(surface, (200, 0, 200), rect)
        pygame.draw.rect(surface, BLACK, rect, 1)


def draw_grid(surface):
    for y in range(GRID_SIZE, SCREEN_HEIGHT - GRID_SIZE, GRID_SIZE):
        for x in range(GRID_SIZE, SCREEN_WIDTH - GRID_SIZE, GRID_SIZE):
            rect = pygame.Rect((x, y), (GRID_SIZE, GRID_SIZE))
            pygame.draw.rect(surface, (60, 60, 60), rect, 1)

def draw_score(surface, score, high_score):
    font = pygame.font.Font(None, FONT_SIZE_MEDIUM)
    score_text = font.render(f'Score: {score}', True, WHITE)
    high_score_text = font.render(f'High Score: {high_score}', True, WHITE)
    surface.blit(score_text, (10, 10))
    surface.blit(high_score_text, (SCREEN_WIDTH - 200, 10))

def draw_sound_status(surface, audio_system):
    font = pygame.font.Font(None, FONT_SIZE_SMALL)
    status = "ON" if audio_system.sound_enabled else "OFF"
    color = GREEN if audio_system.sound_enabled else RED
    
    sound_text = font.render(f'Sound: {status}', True, color)
    surface.blit(sound_text, (SCREEN_WIDTH - 150, SCREEN_HEIGHT - 60))
    
    # Индикатор загрузки звуков
    sounds_loaded = sum([1 for s in [audio_system.eat_sound, audio_system.death_sound, 
                                     audio_system.move_sound, audio_system.menu_sound] if s])
    
    load_text = font.render(f'Sounds: {sounds_loaded}/4', True, 
                          GREEN if sounds_loaded == 4 else YELLOW)
    surface.blit(load_text, (SCREEN_WIDTH - 150, SCREEN_HEIGHT - 30))

def show_game_over(surface, score, high_score, audio_system, death_reason="Вы врезались в стену!"):
    font = pygame.font.Font(None, FONT_SIZE_LARGE)
    small_font = pygame.font.Font(None, FONT_SIZE_MEDIUM)
    
    game_over_text = font.render('GAME OVER', True, RED)
    score_text = small_font.render(f'Final Score: {score}', True, WHITE)
    high_score_text = small_font.render(f'High Score: {high_score}', True, WHITE)
    death_reason_text = small_font.render(death_reason, True, (255, 200, 100))
    restart_text = small_font.render('Press SPACE to restart', True, WHITE)
    menu_text = small_font.render('Press ESC for main menu', True, WHITE)
    sound_hint = small_font.render('Press M to toggle sound', True, (150, 200, 255))
    
    surface.blit(game_over_text, (SCREEN_WIDTH // 2 - game_over_text.get_width() // 2, 
                                SCREEN_HEIGHT // 2 - 180))
    surface.blit(death_reason_text, (SCREEN_WIDTH // 2 - death_reason_text.get_width() // 2, 
                                   SCREEN_HEIGHT // 2 - 110))
    surface.blit(score_text, (SCREEN_WIDTH // 2 - score_text.get_width() // 2, 
                            SCREEN_HEIGHT // 2 - 60))
    surface.blit(high_score_text, (SCREEN_WIDTH // 2 - high_score_text.get_width() // 2, 
                                 SCREEN_HEIGHT // 2 - 10))
    surface.blit(restart_text, (SCREEN_WIDTH // 2 - restart_text.get_width() // 2, 
                              SCREEN_HEIGHT // 2 + 50))
    surface.blit(menu_text, (SCREEN_WIDTH // 2 - menu_text.get_width() // 2, 
                           SCREEN_HEIGHT // 2 + 90))
    surface.blit(sound_hint, (SCREEN_WIDTH // 2 - sound_hint.get_width() // 2, 
                            SCREEN_HEIGHT // 2 + 130))
    
    # Отображаем статус звука
    draw_sound_status(surface, audio_system)

def show_start_screen(surface, audio_system):
    font = pygame.font.Font(None, FONT_SIZE_LARGE)
    small_font = pygame.font.Font(None, FONT_SIZE_MEDIUM)
    
    title_text = font.render('SNAKE GAME', True, GREEN)
    subtitle = small_font.render('with Online Sound Effects', True, (150, 200, 255))
    start_text = small_font.render('Press SPACE to start', True, WHITE)
    controls_text = small_font.render('Use arrow keys to control the snake', True, WHITE)
    warning_text = small_font.render('Avoid walls and yourself!', True, (255, 200, 100))
    sound_hint = small_font.render('Press M to toggle sound effects', True, (150, 200, 255))
    internet_hint = small_font.render('Sounds downloaded from Mixkit.co', True, (200, 200, 100))
    
    surface.blit(title_text, (SCREEN_WIDTH // 2 - title_text.get_width() // 2, 
                            SCREEN_HEIGHT // 2 - 180))
    surface.blit(subtitle, (SCREEN_WIDTH // 2 - subtitle.get_width() // 2, 
                          SCREEN_HEIGHT // 2 - 130))
    surface.blit(start_text, (SCREEN_WIDTH // 2 - start_text.get_width() // 2, 
                            SCREEN_HEIGHT // 2 - 70))
    surface.blit(controls_text, (SCREEN_WIDTH // 2 - controls_text.get_width() // 2, 
                               SCREEN_HEIGHT // 2 - 20))
    surface.blit(warning_text, (SCREEN_WIDTH // 2 - warning_text.get_width() // 2, 
                              SCREEN_HEIGHT // 2 + 30))
    surface.blit(sound_hint, (SCREEN_WIDTH // 2 - sound_hint.get_width() // 2, 
                            SCREEN_HEIGHT // 2 + 80))
    surface.blit(internet_hint, (SCREEN_WIDTH // 2 - internet_hint.get_width() // 2, 
                               SCREEN_HEIGHT // 2 + 120))
    
    # Отображаем статус звука
    draw_sound_status(surface, audio_system)

# Главный управляющий класс
class Game:
    
    def __init__(self):
        print("=" * 60)
        print("ЗМЕЙКА - ВЕРСИЯ С ЗВУКАМИ ИЗ ИНТЕРНЕТА")
        print("=" * 60)
        print("ИГРА БУДЕТ:")
        print("1. Скачивать звуки из интернета (Mixkit.co)")
        print("2. Сохранять их локально для повторного использования")
        print("3. Использовать резервные звуки если интернет недоступен")
        print("=" * 60)
        
        # Инициализируем аудио систему
        print("Пожалуйста, подождите, загружаю звуки...")
        self.audio = AudioSystem()
        
        # Инициализируем игровые объекты
        self.snake = Snake(self.audio)
        self.food = Food()
        self.food.randomize_position(self.snake.positions)
        
        # Игровые переменные
        self.high_score = 0
        self.game_state = "MENU"
        self.speed = INITIAL_SPEED
        self.running = True
        self.death_reason = ""
        
        print("=" * 60)
        print("ИГРА ГОТОВА!")
        print("=" * 60)
        print("ДОСТУПНЫЕ ЗВУКИ ИЗ ИНТЕРНЕТА:")
        print("  - Поедание еды: смешной 'юнилок' звук")
        print("  - Смерть: ретро 'game over' звук")
        print("  - Движение: щелчок шара")
        print("  - Меню: клик выбора")
        print("=" * 60)
        print("УПРАВЛЕНИЕ:")
        print("  ← ↑ → ↓  - движение змейки")
        print("  SPACE    - начать игру / перезапустить")
        print("  ESC      - выйти в меню / выйти из игры")
        print("  M        - вкл/выкл звуки")
        print("=" * 60)
        print("ПРИМЕЧАНИЕ: При первом запуске потребуется интернет")
        print("для скачивания звуков. Они сохранятся локально.")
        print("=" * 60)

    # Главный игровой цикл
    def run(self):
        while self.running:
            # Обработка событий
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    self.running = False
                elif event.type == pygame.KEYDOWN:
                    self.handle_keydown(event)
            
            # Очистка экрана
            screen.fill(BACKGROUND)
            
            # Обновление состояния игры
            self.update_game_state()
            
            # Обновление экрана
            pygame.display.flip()
            clock.tick(self.speed)
        
        pygame.quit()
        sys.exit()

    # Обработка нажатий клавиш
    def handle_keydown(self, event):
        # Глобальная горячая клавиша для звука
        if event.key == pygame.K_m:
            self.audio.toggle_sound()
            # Воспроизводим звук меню при переключении
            self.audio.play_menu_sound()
            return
        
        # Обработка по состоянию игры
        if self.game_state == "MENU":
            if event.key == pygame.K_SPACE:
                self.game_state = "PLAYING"
                print("Игра началась! Удачи!")
                self.audio.play_menu_sound()
            elif event.key == pygame.K_ESCAPE:
                self.running = False
        
        elif self.game_state == "PLAYING":
            if event.key == pygame.K_UP:
                self.snake.turn((0, -1))
            elif event.key == pygame.K_DOWN:
                self.snake.turn((0, 1))
            elif event.key == pygame.K_LEFT:
                self.snake.turn((-1, 0))
            elif event.key == pygame.K_RIGHT:
                self.snake.turn((1, 0))
            elif event.key == pygame.K_ESCAPE:
                self.game_state = "MENU"
                print("Возврат в меню")
                self.audio.play_menu_sound()
        
        elif self.game_state == "GAME_OVER":
            if event.key == pygame.K_SPACE:
                self.restart_game()
            elif event.key == pygame.K_ESCAPE:
                self.game_state = "MENU"
                self.audio.play_menu_sound()

    # Обновление состояния игры в зависимости от текущего режима
    def update_game_state(self):
        if self.game_state == "MENU":
            show_start_screen(screen, self.audio)
        
        elif self.game_state == "PLAYING":
            # Движение змейки и проверка смерти
            if not self.snake.move():
                self.game_state = "GAME_OVER"
                
                # Определяем причину смерти
                head = self.snake.get_head_position()
                if (head[0] < 0 or head[0] >= GRID_WIDTH or 
                    head[1] < 0 or head[1] >= GRID_HEIGHT):
                    self.death_reason = "Вы врезались в стену!"
                    print("Смерть: столкновение со стеной!")
                else:
                    self.death_reason = "Вы врезались в себя!"
                    print("Смерть: столкновение с собой!")
                
                # Обновление рекорда
                if self.snake.score > self.high_score:
                    self.high_score = self.snake.score
                    print(f"Новый рекорд: {self.high_score} очков!")
            
            # Проверка съедания еды 
            elif self.snake.get_head_position() == self.food.position:
                self.snake.grow()  
                self.food.randomize_position(self.snake.positions)
                self.speed += SPEED_INCREMENT
                print(f"Съедена еда! Очки: {self.snake.score}, Скорость: {self.speed:.1f}")
            
            # Отрисовка игровых объектов
            draw_grid(screen)
            self.snake.draw(screen)
            self.food.draw(screen)
            draw_score(screen, self.snake.score, self.high_score)
            draw_sound_status(screen, self.audio)
        
        elif self.game_state == "GAME_OVER":
            show_game_over(screen, self.snake.score, self.high_score, self.audio, self.death_reason)

    # Полный перезапуск игры
    def restart_game(self):
        self.snake.reset()
        self.food.randomize_position(self.snake.positions)
        self.game_state = "PLAYING"
        self.speed = INITIAL_SPEED
        self.death_reason = ""
        print("Игра перезапущена!")
        self.audio.play_menu_sound()

# Запуск игры
if __name__ == "__main__":
    try:
        # Проверяем наличие интернет-соединения
        print("Проверяю подключение к интернету...")
        urllib.request.urlopen("https://www.google.com", timeout=5)
        print("✓ Интернет соединение доступно")
    except:
        print("⚠ Интернет соединение недоступно")
        print("Будут использованы резервные звуки")
    
    game = Game()
    game.run()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.4)
Hello from the pygame community. https://www.pygame.org/contribute.html
Проверяю подключение к интернету...
✓ Интернет соединение доступно
ЗМЕЙКА - ВЕРСИЯ С ЗВУКАМИ ИЗ ИНТЕРНЕТА
ИГРА БУДЕТ:
1. Скачивать звуки из интернета (Mixkit.co)
2. Сохранять их локально для повторного использования
3. Использовать резервные звуки если интернет недоступен
Пожалуйста, подождите, загружаю звуки...
ИНИЦИАЛИЗАЦИЯ АУДИО СИСТЕМЫ
Скачиваю звук eat...
✗ Ошибка скачивания звука eat: HTTP Error 403: Forbidden
Пробую альтернативный URL для eat...
Скачиваю звук eat...
✗ Ошибка скачивания звука eat: HTTP Error 403: Forbidden
Создаю резервный звук для eat...
✓ Создан резервный звук eat
Звук 'eat' готов к использованию
Скачиваю звук death...
✗ Ошибка скачивания звука death: HTTP Error 403: Forbidden
Пробую альтернативный URL для death...
Скачиваю звук death...
✗ Ошибка скачивания звука death: HTTP Error 403: Forbidden
Создаю резервный звук для death...
✓ Создан резервный з

SystemExit: 

C:\ProgramData\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
